# T4-bonus · Foundry deps as code

## Goal

Move the Foundry project, model deployment, and KB creation from notebook
cells into Bicep, with the MCP endpoint registration and its secret
referenced from Key Vault rather than inlined anywhere.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../infra/bicep/modules/foundry.bicep").exists()
assert Path("../infra/bicep/modules/keyvault.bicep").exists()


## Concept

`12`-`15` built Foundry resources interactively to teach the concepts.
This notebook is the reproducibility pass: the same resources, from Bicep,
with the MCP endpoint's auth secret going into Key Vault and referenced
by URI — never printed into a notebook cell or committed to
`agents/contract-renewal-desk` in plaintext.


## Build


In [ ]:
import subprocess, json
deploy = subprocess.run([
    "az", "deployment", "group", "create",
    "--resource-group", "$AZURE_RESOURCE_GROUP",
    "--template-file", "../infra/bicep/main.bicep",
    "--parameters", "namePrefix=crd-dev", "location=$AZURE_LOCATION",
], capture_output=True, text=True)
outputs = json.loads(deploy.stdout)["properties"]["outputs"] if deploy.returncode == 0 else {}
print(outputs)


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")

mcp_server = yaml.safe_load((workspace / "mcp-servers.yaml").read_text())[0]
mcp_server["auth"]["secretRef"] = f"{outputs.get('keyVaultUri', {}).get('value')}secrets/finance-ops-mcp-key"
(workspace / "mcp-servers.yaml").write_text(yaml.dump([mcp_server], sort_keys=False))
print("MCP secret is now a Key Vault reference, not an inline value")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import subprocess
# Re-apply — idempotency check consistent with every other IaC bonus notebook
deploy2 = subprocess.run([
    "az", "deployment", "group", "create",
    "--resource-group", "$AZURE_RESOURCE_GROUP",
    "--template-file", "../infra/bicep/main.bicep",
    "--parameters", "namePrefix=crd-dev", "location=$AZURE_LOCATION",
], capture_output=True, text=True)
assert deploy2.returncode == 0
print("second apply succeeded with no new resources created")


## Cost


In [ ]:
print("Bicep re-apply of existing resources creates nothing new. Foundry compute/storage billing is separate from Copilot Credits — track both.")


## Teardown


In [ ]:
print("No teardown — this notebook only proves the reproducible-Foundry-deps pattern used implicitly by 12-15.")
